In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

# --- STEP 1: Load and Inspect Data ---
print("--- Loading Datasets ---")

movies_df = pd.read_csv('E:\\AI-ML-SQL\\project-2\\movies.csv')
genres_df = pd.read_csv('E:\\AI-ML-SQL\\project-2\\genres.csv')
movie_genres_df = pd.read_csv('E:\\AI-ML-SQL\\project-2\\movie_genres.csv')
cast_df = pd.read_csv('E:\\AI-ML-SQL\\project-2\\cast.csv')
crew_df = pd.read_csv('E:\\AI-ML-SQL\\project-2\\crew.csv')
movie_keywords_df = pd.read_csv('E:\\AI-ML-SQL\\project-2\\movie_keywords.csv')

dfs = {
    'movies': movies_df,
    'genres': genres_df,
    'movie_genres': movie_genres_df,
    'cast': cast_df,
    'crew': crew_df,
    'movie_keywords': movie_keywords_df
}

for name, df in dfs.items():
    print(f"\n=== {name.upper()} ===")
    print(f"Shape: {df.shape}")
    print("Data Types & Null Counts:")
    print(df.info())



--- Loading Datasets ---

=== MOVIES ===
Shape: (2509, 12)
Data Types & Null Counts:
<class 'pandas.DataFrame'>
RangeIndex: 2509 entries, 0 to 2508
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           2509 non-null   int64  
 1   title              2509 non-null   str    
 2   original_language  2509 non-null   str    
 3   release_date       2509 non-null   str    
 4   runtime            2509 non-null   int64  
 5   budget             2509 non-null   float64
 6   revenue            2509 non-null   float64
 7   popularity         2509 non-null   float64
 8   vote_average       2509 non-null   float64
 9   vote_count         2509 non-null   int64  
 10  status             2509 non-null   str    
 11  overview           2509 non-null   str    
dtypes: float64(4), int64(3), str(5)
memory usage: 989.6 KB
None

=== GENRES ===
Shape: (19, 2)
Data Types & Null Counts:
<class 'pandas.DataFrame'

In [2]:
# CLEAN EACH TABLE
print("\n--- Cleaning Data ---")

for name in dfs:
    # 1. Remove duplicates
    before = len(dfs[name])
    dfs[name] = dfs[name].drop_duplicates()
    duplicates_removed = before - len(dfs[name])

    # 2. Identify only text/object columns
    text_cols = dfs[name].select_dtypes(include=["object"]).columns
    
    # 3. Fill nulls only in those text columns
    dfs[name][text_cols] = dfs[name][text_cols].fillna("N/A")

    print(
        f"{name}: Removed {duplicates_removed} duplicate(s) | "
        f"Filled nulls in {len(text_cols)} text column(s)"
    )


--- Cleaning Data ---
movies: Removed 6 duplicate(s) | Filled nulls in 5 text column(s)
genres: Removed 0 duplicate(s) | Filled nulls in 1 text column(s)
movie_genres: Removed 10 duplicate(s) | Filled nulls in 0 text column(s)
cast: Removed 15 duplicate(s) | Filled nulls in 2 text column(s)
crew: Removed 8 duplicate(s) | Filled nulls in 3 text column(s)
movie_keywords: Removed 20 duplicate(s) | Filled nulls in 1 text column(s)


C:\Users\manoj\AppData\Local\Temp\ipykernel_26532\747255534.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = dfs[name].select_dtypes(include=["object"]).columns
C:\Users\manoj\AppData\Local\Temp\ipykernel_26532\747255534.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

In [3]:
# B. Fix Data Types
# Movies table: parse release_date as datetime, ensure IDs/counts are integers
movies_df['release_date'] = pd.to_datetime(
    movies_df['release_date'], errors='coerce'
)
movies_df['movie_id'] = movies_df['movie_id'].astype(int)
movies_df['runtime'] = movies_df['runtime'].fillna(0).astype(int)
movies_df['budget'] = movies_df['budget'].fillna(0).astype(int)
movies_df['revenue'] = movies_df['revenue'].fillna(0).astype(int)
movies_df['vote_count'] = movies_df['vote_count'].fillna(0).astype(int)

# Reference & Bridge/Child tables ID casting
genres_df['genre_id'] = genres_df['genre_id'].astype(int)
movie_genres_df['movie_id'] = movie_genres_df['movie_id'].astype(int)
movie_genres_df['genre_id'] = movie_genres_df['genre_id'].astype(int)

cast_df['movie_id'] = cast_df['movie_id'].astype(int)
cast_df['person_id'] = cast_df['person_id'].astype(int)

crew_df['movie_id'] = crew_df['movie_id'].astype(int)
crew_df['person_id'] = crew_df['person_id'].astype(int)

movie_keywords_df['movie_id'] = movie_keywords_df['movie_id'].astype(int)
movie_keywords_df['keyword_id'] = movie_keywords_df['keyword_id'].astype(int)


# C. Remove Orphaned Rows (dependent table rows whose movie_id is no longer in movies_df)
valid_movie_ids = set(movies_df['movie_id'])

for name, df in [
    ('movie_genres', movie_genres_df),
    ('cast', cast_df),
    ('crew', crew_df),
    ('movie_keywords', movie_keywords_df),
]:
  before = len(df)
  df_cleaned = df[df['movie_id'].isin(valid_movie_ids)]
  print(
      f'Removed {before - len(df_cleaned)} orphaned row(s) from {name}'
  )
  globals()[f'{name}_df'] = df_cleaned

Removed 0 orphaned row(s) from movie_genres
Removed 0 orphaned row(s) from cast
Removed 0 orphaned row(s) from crew
Removed 0 orphaned row(s) from movie_keywords


In [4]:
# 3. HANDLING BUDGET/REVENUE VALUES OF 0
# NOTE: TMDB uses 0 to indicate unknown financial data, not free movies.
# For summary statistics or modeling where 0 skews calculations, replace them with NaN.
print('\n--- Note on Budget/Revenue ---')
print(
    'Budget/Revenue values of 0 are preserved as 0 to denote missing/unknown data'
    ' per TMDB standards, or can be converted to NaN during specific Pandas EDA steps.'
)


--- Note on Budget/Revenue ---
Budget/Revenue values of 0 are preserved as 0 to denote missing/unknown data per TMDB standards, or can be converted to NaN during specific Pandas EDA steps.


In [5]:
# 4. LOAD CLEANED TABLES INTO A SQL DATABASE (MySQL using SQLAlchemy)
print('\n--- Loading into MySQL Database ---')

# Replace username, password, host, and database_name with your actual MySQL config
DB_USER = 'root'
DB_PASSWORD = 'Manoj1297%40'
DB_HOST = '127.0.0.1'
DB_NAME = 'tmdb'

# Create connection engine using PyMySQL driver
engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}'
)

# Sync your individual variables with the cleaned dfs dictionary first
movies_df = dfs['movies']
genres_df = dfs['genres']
movie_genres_df = dfs['movie_genres']
cast_df = dfs['cast']
crew_df = dfs['crew']
movie_keywords_df = dfs['movie_keywords']

# 1. Temporarily disable foreign key checks
with engine.begin() as connection:
  connection.execute(
      __import__('sqlalchemy').text('SET FOREIGN_KEY_CHECKS = 0;')
  )

# 2. Reload all tables using pandas to_sql safely
movies_df.to_sql('movies', engine, if_exists='replace', index=False)
genres_df.to_sql('genres', engine, if_exists='replace', index=False)
movie_genres_df.to_sql('movie_genres', engine, if_exists='replace', index=False)
cast_df.to_sql('cast', engine, if_exists='replace', index=False)
crew_df.to_sql('crew', engine, if_exists='replace', index=False)
movie_keywords_df.to_sql('movie_keywords', engine, if_exists='replace', index=False)

# 3. Re-enable foreign key checks
with engine.begin() as connection:
  connection.execute(
      __import__('sqlalchemy').text('SET FOREIGN_KEY_CHECKS = 1;')
  )

print('All tables successfully replaced and reloaded!')


--- Loading into MySQL Database ---
All tables successfully replaced and reloaded!


In [6]:
# 5. RUN A SIMPLE TEST QUERY TO CONFIRM PROPER LOADING
print('\n--- Running Test SQL Query ---')
test_query = """
    SELECT m.title, g.genre_name, m.release_date
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    LIMIT 5;
"""

test_results = pd.read_sql(test_query, engine)
print(test_results)

print('\nMySQL database successfully populated and verified!')





--- Running Test SQL Query ---
          title       genre_name release_date
0  Interstellar  Science Fiction   2014-11-05
1  Interstellar            Drama   2014-11-05
2  Interstellar        Adventure   2014-11-05
3     Inception        Adventure   2010-07-15
4     Inception  Science Fiction   2010-07-15

MySQL database successfully populated and verified!
